# ⚛️ Hackathon Algorithmique Quantique : Optimisation de l'Énergie Moléculaire

Bienvenue dans ce challenge ! L'objectif de ce hackathon est de trouver l'énergie de l'état fondamental de différentes molécules à l'aide d'algorithmes quantiques variationnels (comme le VQE). 

Dans ce notebook, nous allons :
1. Définir nos molécules (Chaîne d'hydrogène et Hydrure de Lithium - LiH).
2. Générer les représentations fermioniques de ces molécules, avec et sans "Frozen Core" (gel du cœur).
3. Établir une ligne de base (Baseline) avec un VQE classique (Ansatz UCCSD).
4. **À vous de jouer :** Compléter la fonction `run()` pour battre notre temps d'exécution tout en conservant (ou en améliorant) la précision chimique de l'énergie !

In [ ]:
import time
import numpy as np

# Qiskit et Qiskit Nature
from qiskit_nature.second_q.formats.molecule_info import MoleculeInfo
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.transformers import FreezeCoreTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper, ParityMapper
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD

from qiskit_algorithms import VQE, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import SLSQP
from qiskit.primitives import StatevectorEstimator as Estimator

## 1. Définition de la Molécule

Nous utilisons la classe `MoleculeInfo` pour définir la géométrie de nos molécules. 
Nous vous fournissons une fonction pour générer une chaîne d'atomes d'Hydrogène, ainsi que la configuration pour la molécule d'Hydrure de Lithium (LiH) qui servira pour notre calcul de référence.

In [ ]:
def build_hydrogen_chain(num_atoms, bond_distance):
    symbols = ["H"] * num_atoms
    coords = [(i * bond_distance, 0.0, 0.0) for i in range(num_atoms)]
    multiplicity = (num_atoms % 2) + 1 
    return MoleculeInfo(symbols=symbols, coords=coords, multiplicity=multiplicity, charge=0)

# Définition de H2 (chaine de 2 atomes)
h2_info = build_hydrogen_chain(num_atoms=2, bond_distance=0.735)

# Définition de la molécule LiH pour notre Baseline
lih_info = MoleculeInfo(
    symbols=["Li", "H"],
    coords=[(0.0, 0.0, 0.0), (0.0, 0.0, 1.5474)],
    multiplicity=1,
    charge=0
)

## 2. Génération de la molécule (PySCF) & Frozen Core

Nous utilisons le driver `PySCF` pour calculer les intégrales à 1 et 2 corps. 
Afin de vous permettre de vous entraîner sur des tailles d'espace de Hilbert différentes mais chimiquement identiques, nous générons deux versions du problème LiH :
- **Complète** : Tous les électrons sont pris en compte.
- **Gelée (Frozen Core)** : Les électrons de cœur (qui participent très peu aux liaisons chimiques) sont ignorés, réduisant ainsi drastiquement le nombre de qubits nécessaires.

In [ ]:
# --- Molécule sans geler le coeur ---
driver_full = PySCFDriver.from_molecule(lih_info, basis="sto3g")
problem_full = driver_full.run()
fermionic_op_full = problem_full.hamiltonian.second_q_op()

# --- Molécule en gelant le coeur (Frozen Core) ---
driver_frozen = PySCFDriver.from_molecule(lih_info, basis="sto3g")
problem_frozen = driver_frozen.run()

# Application du FreezeCoreTransformer
transformer = FreezeCoreTransformer(freeze_core=True)
problem_frozen = transformer.transform(problem_frozen)
fermionic_op_frozen = problem_frozen.hamiltonian.second_q_op()

print(f"Nombre d'orbitales spatiales (Full) : {problem_full.num_spatial_orbitals}")
print(f"Nombre d'orbitales spatiales (Frozen) : {problem_frozen.num_spatial_orbitals}")

## 3. Calcul des valeurs propres (Énergie Exacte)

Avant de lancer nos algorithmes quantiques, nous devons connaître la "vraie" réponse. Nous utilisons un solveur classique (`NumPyMinimumEigensolver`) pour diagonaliser exactement le Hamiltonien et trouver l'énergie de l'état fondamental de notre molécule LiH gelée. 
Cette valeur servira à calculer la précision de vos méthodes.

In [ ]:
mapper = JordanWignerMapper()
exact_solver = NumPyMinimumEigensolver()
exact_calc = GroundStateEigensolver(mapper, exact_solver)

exact_result_frozen = exact_calc.solve(problem_frozen)
exact_energy_frozen = exact_result_frozen.total_energies[0]

exact_result_full = exact_calc.solve(problem_full)
exact_energy_full = exact_result_full.total_energies[0]

print(f"Énergie totale exacte de l'état fondamental (LiH gelé) : {exact_energy_frozen:.6f} Hartree")

print(f"Énergie totale exacte de l'état fondamental (LiH gelé) : {exact_energy_full:.6f} Hartree")

## 4. Baseline VQE UCCSD

Nous lançons maintenant un VQE classique sur la molécule LiH gelée.
- **Ansatz** : UCCSD (Unitary Coupled-Cluster Singles and Doubles).
- **État initial** : Tous les paramètres fixés à 0 (partant de l'état Hartree-Fock).
- **Objectif** : Mesurer le temps d'exécution (`t_ref`) et vérifier l'énergie. **C'est ce temps que vous devrez battre !**

In [ ]:
# Configuration de l'état de référence Hartree-Fock
initial_state = HartreeFock(
    problem_frozen.num_spatial_orbitals,
    problem_frozen.num_particles,
    mapper
)

# Configuration de l'Ansatz UCCSD
ansatz = UCCSD(
    problem_frozen.num_spatial_orbitals,
    problem_frozen.num_particles,
    mapper,
    initial_state=initial_state
)

# Initialisation de tous les paramètres à 0
initial_pt = np.zeros(ansatz.num_parameters)

# Setup du VQE
estimator = Estimator()
optimizer = SLSQP(maxiter=100)
vqe = VQE(estimator, ansatz, optimizer, initial_point=initial_pt)
vqe_solver = GroundStateEigensolver(mapper, vqe)

print("Lancement du VQE UCCSD de référence. Veuillez patienter...")
start_time = time.time()
baseline_result = vqe_solver.solve(problem_frozen)
end_time = time.time()

t_ref = end_time - start_time
baseline_energy = baseline_result.total_energies[0]
precision_baseline = abs(exact_energy - baseline_energy)

print("-" * 50)
print(f"⏱️ Temps de référence (t_ref) : {t_ref:.2f} secondes")
print(f"⚛️ Énergie calculée (Baseline) : {baseline_energy:.6f} Hartree")
print(f"🎯 Précision (Erreur) absolue : {precision_baseline:.6f} Hartree")
print("-" * 50)

## 5. Le Hackathon : Votre fonction `run()`

C'est ici que vous intervenez. Vous devez implémenter la fonction `run()`. 

**Instructions :**
1. Elle prend en entrée l'objet `problem` (pour récupérer le nombre de particules/orbitales si besoin) et l'opérateur `fermionic_op` (sortie du PySCF driver).
2. **Mapper** : Appliquez le mapper de votre choix (`ParityMapper`, `JordanWignerMapper`, `BravyiKitaevMapper`...) pour transformer la représentation fermionique sous la forme d'une somme de matrices de Pauli (qubit operator).
3. **Méthode** : Configurez votre propre ansatz, état initial (ex: paramétrage aléatoire ou hardware-efficient) et optimiseur.
4. **Retour** : La fonction doit retourner l'énergie totale (nucléaire incluse) trouvée et le temps d'exécution.

> 💡 *Astuce : Pour gagner en temps, explorez des ansatz plus légers que UCCSD (comme `EfficientSU2`), des mappers qui réduisent le nombre de qubits (comme `ParityMapper` avec réduction de symétrie), ou des optimiseurs plus agressifs.*

In [ ]:
def run(problem, fermionic_op):
    """
    Exécute votre algorithme quantique sur l'opérateur fermionique donné.
    """
    # On démarre le chrono !
    start_time = time.time()
    
    # ==========================================
    # 🛠️ VOTRE CODE COMMENCE ICI 🛠️
    # ==========================================
    
    # 1. Choisissez et appliquez un mapper
    # my_mapper = ... 
    # qubit_op = my_mapper.map(fermionic_op)
    
    # 2. Définissez votre Ansatz et votre état initial
    # ...
    
    # 3. Configurez votre VQE (ou autre algorithme quantique)
    # ...
    
    # 4. Calculez l'énergie fondamentale
    # N'oubliez pas d'ajouter l'énergie de répulsion nucléaire si vous 
    # calculez manuellement l'énergie électronique à partir de l'opérateur.
    # (Ou utilisez directement GroundStateEigensolver)
    
    my_energy = 0.0 # Remplacez par votre résultat (Énergie totale)
    
    # ==========================================
    # 🏁 VOTRE CODE S'ARRÊTE ICI 🏁
    # ==========================================
    
    end_time = time.time()
    execution_time = end_time - start_time
    
    return my_energy, execution_time

## 6. Évaluation

Testons votre algorithme sur la molécule LiH gelée et comparons vos résultats avec la Baseline !

In [ ]:
print("Lancement de votre fonction run()...")

# Exécution de votre fonction sur le problème gelé
participant_energy, participant_time = run(problem_frozen, fermionic_op_frozen)

# Calcul des métriques
participant_error = abs(exact_energy - participant_energy)
speedup = t_ref / participant_time if participant_time > 0 else float('inf')

print("\n" + "="*50)
print("🏆 RÉSULTATS DU HACKATHON 🏆")
print("="*50)
print(f"⏱️ Votre temps       : {participant_time:.2f} secondes (x{speedup:.2f} par rapport au t_ref)")
print(f"⚛️ Votre énergie     : {participant_energy:.6f} Hartree")
print(f"🎯 Votre précision   : {participant_error:.6f} Hartree d'erreur")
print("-" * 50)

# Conditions de victoire basiques (à adapter)
CHEMICAL_ACCURACY = 0.0016  # Précision chimique standard (~1 kcal/mol)

if participant_error <= CHEMICAL_ACCURACY:
    print("✅ Précision chimique ATTEINTE !")
else:
    print("❌ Précision chimique NON ATTEINTE. Essayez d'améliorer votre Ansatz ou votre optimiseur.")